# 09 · Watch an API model play the grid game

Run any model from **OpenRouter** (or OpenAI/DeepSeek/Anthropic direct) on the grid
navigation game through the unified interface, then **watch the rollout animate inline**.
No GPU, no weights — just an API key. The model is driven one tick at a time via
`process(EnvironmentStep)`; we capture each rendered frame and play it back.

**Prereq:** an OpenRouter key. Either `export OPENROUTER_API_KEY=...` before launching
Jupyter, or drop it in a file at the project root (this cell looks for it).


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))   # unified/ on path
# Load the key: prefer the env var; else read a file at the repo root.
if not os.environ.get('OPENROUTER_API_KEY'):
    for p in ['../../OPENROUTER.API', os.path.expanduser('~/Brain-Score Unified/OPENROUTER.API')]:
        if os.path.exists(p):
            os.environ['OPENROUTER_API_KEY'] = open(p).read().strip(); break
assert os.environ.get('OPENROUTER_API_KEY'), 'Set OPENROUTER_API_KEY (env var or repo-root file).'
print('key loaded:', os.environ['OPENROUTER_API_KEY'][:8] + '...')


## Build the model

`obs_mode='ascii'` lets the model read a perfect text board (text models can play, and even
vision models reason better this way — DeepSeek-R1 on ASCII beat the VLMs in the original game).
Switch to `obs_mode='vision'` to test perception from the rendered frame instead.
`MODEL` is any OpenRouter id — swap freely (`anthropic/claude-3.5-sonnet`, `deepseek/deepseek-chat`, ...).


In [ ]:
from brainscore_core.model_interface import BrainScoreModel
from brainscore.model_helpers.api_behavioral import build_api_action_fn
from brainscore.harnesses.grid_game import GridGameEnv

MODEL    = 'openai/gpt-4o-mini'   # any OpenRouter model id
OBS_MODE = 'ascii'                # 'ascii' (reads text board) or 'vision' (reads the frame)

action_fn = build_api_action_fn('openrouter', MODEL, obs_mode=OBS_MODE, max_tokens=300)
model = BrainScoreModel(identifier=f'openrouter:{MODEL}', model=None,
                        region_layer_map={}, preprocessors={}, action_fn=action_fn)
print('model ready:', model.identifier)


## Play one episode (capturing every frame)

`GridGameEnv` renders an `(H,W,3)` frame each tick. We drive the loop and keep the frames
+ the move the model chose. Each tick is one API call, so this is sequential — a 6x6 board
solves in a handful of moves.


In [ ]:
import numpy as np
from brainscore_core.model_interface import EnvironmentStep

def rollout(model, env, max_steps=20):
    obs = env.reset()
    frames, actions, labels = [env.render()], [], []
    step = EnvironmentStep(observation=obs, instruction=obs['instruction'], is_first=True, step_num=0)
    info = {}
    for t in range(max_steps):
        resp = model.process(step)
        a = int(np.asarray(resp.action).reshape(-1)[0])
        actions.append(a); labels.append(obs['legal_actions'].get(a, str(a)))
        obs, reward, done, info = env.step(a)
        frames.append(env.render())
        if done: break
        step = EnvironmentStep(observation=obs, instruction=obs['instruction'], step_num=t+1, reward=reward)
    return frames, actions, labels, bool(info.get('reached_goal', False))

env = GridGameEnv(size=6, n_walls=0, seed=1, max_steps=20)
frames, actions, labels, solved = rollout(model, env)
print(f'solved={solved}  steps={len(actions)}  moves={labels}')


## Watch it play

Inline animation — play / pause / scrub the rollout. Blue = player, green = goal.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(3.4, 3.4)); plt.close(fig)
im = ax.imshow(frames[0]); ax.axis('off')
def update(i):
    im.set_data(frames[i])
    if i == 0:
        ax.set_title('start', fontsize=11)
    elif solved and i == len(frames) - 1:
        ax.set_title(f'move {i}: {labels[i-1]}  —  SOLVED ✅', fontsize=11)
    else:
        ax.set_title(f'move {i}: {labels[i-1]}', fontsize=11)
    return [im]
anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=700, blit=False)
HTML(anim.to_jshtml())


## Try more

- **Swap the model:** set `MODEL` to any OpenRouter id (`anthropic/claude-3.5-sonnet`,
  `deepseek/deepseek-chat`, `google/gemini-2.0-flash-001`, ...). Catalog: https://openrouter.ai/models
- **Test perception:** set `OBS_MODE='vision'` — the model reads the rendered frame instead of the
  text board. Absolute-action grids are easier than MiniGrid's egocentric ones, but small models still struggle.
- **Harder boards:** `GridGameEnv(size=8, n_walls=4, seed=...)`.
- **A guaranteed-perfect run to watch:** use the scripted oracle instead of an API model —
  `from brainscore.harnesses.grid_game import greedy_oracle_policy, make_model; model = make_model(greedy_oracle_policy)`.
- **Score it (no animation):** `python ../scripts/vlm_game/play_api_game.py --provider openrouter --model <id> --obs_mode ascii --games 15`.
